In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from dataclasses import dataclass

@dataclass
class ColourContext:
    favourite_colour: str = "blue"
    least_favourite_colour: str = "yellow"

In [3]:
from langchain.agents import create_agent

agent = create_agent(
    model="gpt-5-nano",
    context_schema=ColourContext  
)

In [4]:
from langchain.messages import HumanMessage

response = agent.invoke(
    {"messages": [HumanMessage(content="What is my favourite colour?")]},
    context=ColourContext()
)

In [5]:
from pprint import pprint

pprint(response)

{'messages': [HumanMessage(content='What is my favourite colour?', additional_kwargs={}, response_metadata={}, id='3ac6ae1a-ea1d-4c65-a63d-a2652c0c16c9'),
              AIMessage(content="I don’t know your favourite colour—I've no way to access your thoughts. If you’d like, I can guess or you can tell me.\n\nQuick guess: blue. It’s a very common favourite.\n\nWant me to guess more accurately? Answer a couple of quick questions:\n- Do you prefer warm colors (reds, oranges, yellows) or cool colors (blues, greens, purples)?\n- Do you like bright, saturated colors or muted tones?\n- Do you tend to wear or decorate with colors found in nature (sky/sea/earth) or something else?", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 832, 'prompt_tokens': 12, 'total_tokens': 844, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 704, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tok

## Accessing Context

In [6]:
from langchain.tools import tool, ToolRuntime

@tool
def get_favourite_colour(runtime: ToolRuntime) -> str:
    """Get the favourite colour of the user"""
    return runtime.context.favourite_colour

@tool
def get_least_favourite_colour(runtime: ToolRuntime) -> str:
    """Get the least favourite colour of the user"""
    return runtime.context.least_favourite_colour

In [7]:
agent = create_agent(
    model="gpt-5-nano",
    tools=[get_favourite_colour, get_least_favourite_colour],
    context_schema=ColourContext
)

In [8]:
response = agent.invoke(
    {"messages": [HumanMessage(content="What is my favourite colour?")]},
    context=ColourContext()
)

pprint(response)

{'messages': [HumanMessage(content='What is my favourite colour?', additional_kwargs={}, response_metadata={}, id='285ee0b1-bce6-4732-80bc-ea64110aa31e'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 213, 'prompt_tokens': 149, 'total_tokens': 362, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 192, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DVmx8Iw6yxFDvpLEMESO4mSeitnjU', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019d9dc9-6257-77e0-aa98-e32890e30832-0', tool_calls=[{'name': 'get_favourite_colour', 'args': {}, 'id': 'call_pZI9kGbQ6gmOgHpnhnnyz1dE', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 149, 'output_

In [10]:
response = agent.invoke(
    {"messages": [HumanMessage(content="What is my favourite colour?")]},
    context=ColourContext(favourite_colour="green")
)

print(response["messages"][-1].content)

Your favourite colour is green. Would you like me to remember that for future chats or update it if it changes?


Your favourite colour is blue. Would you like me to remember that for future chats or help with anything else related to colours?


In [11]:
response = agent.invoke(
    {"messages": [HumanMessage(content="What is my favourite colour?")]},
    context=ColourContext()
)

print(response["messages"][-1].content)


Your favourite colour is blue.

Would you like me to update it or use it for something else?
